In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00


In [2]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("FP16 Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

FP16 Model Loaded


In [4]:
prompt = "Explain artificial intelligence in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

start = time.time()
output = model_fp16.generate(**inputs, max_new_tokens=100)
end = time.time()

print("FP16 Time:", end - start)

FP16 Time: 0.9263055324554443


In [11]:
bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_int8,
    device_map="auto"
)

print("INT8 Model Loaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT8 Model Loaded


In [12]:
start = time.time()
output = model_int8.generate(**inputs, max_new_tokens=100)
end = time.time()

print("INT8 Time:", end - start)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


INT8 Time: 1.0582938194274902


In [13]:
model_int8.save_pretrained("quantized/model-int8")
tokenizer.save_pretrained("quantized/model-int8")

print("INT8 Model Saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 Model Saved


In [15]:
bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_int4,
    device_map="auto"
)

print("INT4 Model Loaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT4 Model Loaded


In [16]:
start = time.time()
output = model_int4.generate(**inputs, max_new_tokens=100)
end = time.time()

print("INT4 Time:", end - start)

INT4 Time: 0.0877223014831543


In [17]:
model_int4.save_pretrained("quantized/model-int4")
tokenizer.save_pretrained("quantized/model-int4")

print("INT4 Model Saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 Model Saved


In [ ]:
!zip -r quantized.zip quantized
from google.colab import files
files.download("quantized.zip")

  adding: quantized/ (stored 0%)
  adding: quantized/model-int8/ (stored 0%)
  adding: quantized/model-int8/generation_config.json (deflated 29%)
  adding: quantized/model-int8/model.safetensors


zip error: Interrupted (aborting)


FileNotFoundError: Cannot find file: quantized.zip

In [5]:
model_fp16.save_pretrained("model-fp16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!apt update
!apt install -y cmake build-essential
!pip install huggingface_hub

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,797 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 81409, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 81409 (delta 36), reused 3 (delta 3), pack-reused 81341 (from 4)
Receiving objects: 100% (81409/81409), 307.18 MiB | 11.40 MiB/s, done.
Resolving deltas: 100% (58852/58852), done.
Updating files: 100% (2283/2283), done.
/content/llama.cpp


In [ ]:
!cmake -B build
!cmake --build build --config Release

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    local_dir="model-fp16",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/content/llama.cpp/model-fp16'

In [ ]:
!python convert_hf_to_gguf.py model-fp16 --outfile model-f16.gguf --outtype f16

INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_out

In [ ]:
!./build/bin/llama-quantize model-f16.gguf model-q4_0.gguf q4_0

main: build = 8193 (ecd99d6a9)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-f16.gguf' to 'model-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 45 key-value pairs and 201 tensors from model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                            general.license str              = apache-2.0
llama_model_loader: - kv   5:                      general.dataset.count u32              = 4
llama_model_loader: - kv   6:                 

In [ ]:
!mkdir -p quantized
!mv model-q4_0.gguf quantized/

In [ ]:
!mv quantized/model-q4_0.gguf quantized/model.gguf

In [ ]:
#to check speed of model.gguf
!./build/bin/llama-cli -m quantized/model.gguf -p "Explain why the sky is blue." -n 100


Loading model... |-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8193-ecd99d6a9
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Explain why the sky is blue.

|-\|/-\|/ Sure! The sky is blue because it consists of the colors of the visible light spectrum. The colors are made up of wavelengths ranging from 400 nm to 700 nm. The blue color of the sky is primarily caused by the wavelengths of light that are blue to violet in the electromagnetic spectrum. The blue light is emitted by the atmospheric molecules and photons that are present in the atmosphere, and

In [ ]:
!mv /content/llama.cpp/quantized/model.gguf /content/quantized/

In [23]:
import os

def get_folder_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total / (1024**3)

def get_file_size(path):
    return os.path.getsize(path) / (1024**3)

print("FP16 Size:", get_folder_size("/content/drive/MyDrive/model-fp16"), "GB")
print("INT8 Size:", get_folder_size("/content/drive/MyDrive/quantized/model-int8"), "GB")
print("INT4 Size:", get_folder_size("/content/drive/MyDrive/quantized/model-int4"), "GB")
print("GGUF Q4_0 Size:", get_file_size("/content/drive/MyDrive/quantized/model.gguf"), "GB")

FP16 Size: 2.049021898768842 GB
INT8 Size: 1.1515486389398575 GB
INT4 Size: 0.7553469445556402 GB
GGUF Q4_0 Size: 0.592998743057251 GB


In [7]:
def generate_output(model, tokenizer, prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(result)
    print("\n" + "="*60 + "\n")

In [10]:
print("FP16 OUTPUT:\n")

prompt = "<|user|>\nExplain why the sky is blue.\n<|assistant|>\n"

generate_output(model_fp16, tokenizer, prompt)

FP16 OUTPUT:

The sky is blue because the colors in the sky are composed of light in different wavelengths. Blue light, which makes up the majority of the colors seen in the sky, is closest in wavelength to the violet color in the human eye. As the light travels from the sun to the earth, it passes through the atmosphere, which absorbs some of the light in the blue wavelengths. The blue light is then reflected back to our eyes, giving the impression of a blue sky. Additionally, the Earth's atmosphere contains molecules that absorb the blue light and emit the vibrant red and orange colors seen in the sky. The atmosphere also contains other colors, such as green and yellow, but




In [14]:
print("INT8 OUTPUT:\n")
prompt = "<|user|>\nExplain why the sky is blue.\n<|assistant|>\n"
generate_output(model_int8, tokenizer, prompt)

INT8 OUTPUT:

The sky is blue because it contains many different colors of light that are reflected and combined in a specific way. The colors of the sky are primarily due to the combination of blue, green, and red light.
Blue light, which is the light closest to the observer's eye, is the light that reaches us first. It is made up of violet and purple wavelengths, which are absorbed by water molecules in the atmosphere, and reflected back by the earth's atmosphere. This light is responsible for the blue color of the sky.
Green light, which is the light further away from the observer's eye, is responsible for the green color of the sky. It is made up of yellow and orange




In [19]:
print("INT4 OUTPUT:\n")
prompt = "<|user|>\nExplain why the sky is blue.\n<|assistant|>\n"
generate_output(model_int4, tokenizer, prompt)

INT4 OUTPUT:

The sky is blue because the sky is transparent. The blue color of the sky results from the reflected light of the sun, which is scattered by particles in the Earth's atmosphere, such as dust and gas. The particles scatter and reflect the light in a process called absorption, resulting in the blue color of the sky. The scattering and reflection of light can also cause the sky to appear blue, depending on the angle of view, latitude, and atmospheric conditions. Over time, the blue color can fade, but it is typically permanent and cannot be changed by human intervention. The blue sky is a beautiful and awe-inspiring sight that provides an important visual contrast to the horizon, which is usually dark and gray




In [ ]:
!mkdir -p /content/drive/MyDrive/quantized
!mv /content/quantized/model.gguf /content/drive/MyDrive/quantized/model.gguf

In [ ]:
!cp -r /content/quantized/model-int8 /content/drive/MyDrive/llama.cpp/

In [ ]:
!mv /content/quantized/model-int4 /content/drive/MyDrive/quantized/model-int4/
!mv /content/quantized/model-int8 /content/drive/MyDrive/quantized/model-int8/
!mv /content/model-fp16/ /content/drive/MyDrive/model-fp16/
